### Import libraries

In [0]:
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)

import json

### Set variables

In [0]:
BASE_URL = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk"

HISTORICAL_URL = f"{BASE_URL}/HentOffentligData"
AREAS_URL = f"{BASE_URL}/HentOmråder"

### Common functions

In [0]:
def get_nve_data(url: str) -> list | dict:
    """
    Fetch JSON data from the NVE Magasinstatistikk API.
    """
    response = requests.get(
        url,
        headers={"Accept": "application/json"},
        timeout=30,
    )

    response.raise_for_status()

    return response.json()

### Main code

In [0]:
# ---------------------------------------------------------
# 1. Historical reservoir statistics
# ---------------------------------------------------------

reservoir_data = get_nve_data(HISTORICAL_URL)

reservoir_schema = StructType([
    StructField("dato_Id", StringType(), True),
    StructField("omrType", StringType(), True),
    StructField("omrnr", IntegerType(), True),
    StructField("iso_aar", IntegerType(), True),
    StructField("iso_uke", IntegerType(), True),
    StructField("fyllingsgrad", DoubleType(), True),
    StructField("kapasitet_TWh", DoubleType(), True),
    StructField("fylling_TWh", DoubleType(), True),
    StructField("neste_Publiseringsdato", StringType(), True),
    StructField("fyllingsgrad_forrige_uke", DoubleType(), True),
    StructField("endring_fyllingsgrad", DoubleType(), True),
])

df_reservoir = spark.createDataFrame(
    reservoir_data,
    schema=reservoir_schema,
)

df_reservoir = (
    df_reservoir
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "source",
        F.lit("NVE_MAGASINSTATISTIKK")
    )
)

In [0]:
# ---------------------------------------------------------
# 2. Area metadata
# ---------------------------------------------------------

areas_data = get_nve_data(AREAS_URL)

area_rows = []

# API returns:
# [
#     {
#         "land": [...],
#         "elspot": [...],
#         "vassdrag": [...]
#     }
# ]

for area_group in areas_data:
    for area_category, areas in area_group.items():

        for area in areas:
            area_rows.append({
                "area_category": area_category,
                "navn": area.get("navn"),
                "navn_langt": area.get("navn_langt"),
                "beskrivelse": area.get("beskrivelse"),
                "omrType": area.get("omrType"),
                "omrnr": area.get("omrnr"),
            })

df_areas = spark.createDataFrame(area_rows)

In [0]:
# ---------------------------------------------------------
# 3. Write Bronze Delta tables
# ---------------------------------------------------------
spark.sql("""
          CREATE SCHEMA IF NOT EXISTS dbw_dariusz.bronze
          """)
(
    df_reservoir
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.nve_reservoir_statistics")
)


(
    df_areas
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.nve_reservoir_areas")
)